# 03. Zero-shot 표 모델 평가 하네스

## 목표
여러 새 과제에서 정확도, Brier score, 문맥 크기와 행 순열 안정성을 함께 평가합니다. 단일 평균 점수만으로 모델을 판단하지 않는 연습입니다.

In [ ]:
# 02 노트북과 독립 실행할 수 있도록 필요한 함수를 다시 정의합니다.
from math import sqrt
from random import Random

def make_task(seed, rows=100):
    rng = Random(seed)
    wx, wz, bias = rng.uniform(-2, 2), rng.uniform(-2, 2), rng.uniform(-0.5, 0.5)
    result = []
    for _ in range(rows):
        x = rng.gauss(0, 1)
        z = 0.7 * x + rng.gauss(0, 0.8)
        label = int(wx * x + wz * z + bias + rng.gauss(0, 0.25) > 0)
        result.append(({"x": x, "z": z}, label))
    return result

def d2(a, b):
    return sum((a[key] - b[key]) ** 2 for key in a)

def probability(context, row, neighbors=7):
    nearest = sorted(context, key=lambda item: d2(item[0], row))[:neighbors]
    weights = [1 / (sqrt(d2(features, row)) + 1e-6) for features, _ in nearest]
    return sum(w * label for w, (_, label) in zip(weights, nearest)) / sum(weights)

In [ ]:
def metrics(task, context_size, shuffle_seed=None):
    context = task[:context_size]
    if shuffle_seed is not None:
        context = context.copy()
        Random(shuffle_seed).shuffle(context)
    test = task[context_size:]
    probabilities = [probability(context, row) for row, _ in test]
    labels = [label for _, label in test]
    accuracy = sum((p >= 0.5) == y for p, y in zip(probabilities, labels)) / len(labels)
    brier = sum((p - y) ** 2 for p, y in zip(probabilities, labels)) / len(labels)
    return accuracy, brier, probabilities

for size in (10, 20, 40, 60):
    scores = [metrics(make_task(seed), size)[:2] for seed in range(10)]
    mean_accuracy = sum(score[0] for score in scores) / len(scores)
    mean_brier = sum(score[1] for score in scores) / len(scores)
    print(f"context={size:2d} accuracy={mean_accuracy:.3f} brier={mean_brier:.3f}")

In [ ]:
task = make_task(42)
base = metrics(task, 40)[2]
shuffled = metrics(task, 40, shuffle_seed=7)[2]
max_difference = max(abs(a - b) for a, b in zip(base, shuffled))
print(f"행 순열 후 최대 확률 차이: {max_difference:.12f}")
assert max_difference < 1e-12
# 실제 TabFM에도 행·열 순열, 결측, 범주 재인코딩에 대한 metamorphic test를 적용할 수 있습니다.

## 전문가 확장 과제

1. 과제별 정확도의 최솟값과 표준편차를 추가하세요.
2. class imbalance가 큰 과제에서 balanced accuracy와 log loss를 계산하세요.
3. zero-shot 모델과 데이터셋별로 tuning한 baseline의 총 시간·메모리를 함께 기록하세요.
4. subgroup별 calibration과 오류 비용을 평가하세요.
5. 여러 모델의 데이터셋별 승패로 간단한 Elo rating을 구현하되, 신뢰구간과 데이터셋 수를 함께 보고하세요.